# Sensory Composer — Interactive Voila Demo

Run this notebook via Voila to expose an interactive UI for audio feature exploration.

```bash
pip install voila ipywidgets
voila voila_app.ipynb --port=8866
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from scipy.fft import rfft, rfftfreq

SAMPLE_RATE = 44100

In [ ]:
freq_slider = widgets.FloatSlider(
    value=440.0, min=20.0, max=4000.0, step=10.0,
    description='Frequency (Hz):', style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

noise_slider = widgets.FloatSlider(
    value=0.05, min=0.0, max=1.0, step=0.01,
    description='Noise level:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

output = widgets.Output()

def update_plot(freq, noise):
    t = np.linspace(0, 0.05, int(SAMPLE_RATE * 0.05), endpoint=False)
    signal = np.sin(2 * np.pi * freq * t) + noise * np.random.randn(len(t))
    full_t = np.linspace(0, 1.0, SAMPLE_RATE, endpoint=False)
    full_signal = np.sin(2 * np.pi * freq * full_t) + noise * np.random.randn(SAMPLE_RATE)

    window = np.hanning(len(full_signal))
    spectrum = rfft(full_signal * window)
    freqs = rfftfreq(len(full_signal), d=1.0 / SAMPLE_RATE)
    mags = np.abs(spectrum)

    rms = np.sqrt(np.mean(full_signal ** 2))
    centroid = np.sum(freqs * mags) / (np.sum(mags) + 1e-12)

    with output:
        output.clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        axes[0].plot(t, signal, color='#4c6ef5', linewidth=0.8)
        axes[0].set_title('Waveform (50 ms)')
        axes[0].set_xlabel('Time (s)')
        axes[0].set_ylabel('Amplitude')

        axes[1].plot(freqs[:2000], mags[:2000], color='#f783ac', linewidth=0.8)
        axes[1].axvline(x=freq, color='gold', linestyle='--', label=f'Target: {freq:.0f} Hz')
        axes[1].axvline(x=centroid, color='cyan', linestyle=':', label=f'Centroid: {centroid:.0f} Hz')
        axes[1].set_title('FFT Spectrum (0–2 kHz)')
        axes[1].set_xlabel('Frequency (Hz)')
        axes[1].set_ylabel('Magnitude')
        axes[1].legend(fontsize=8)

        plt.suptitle(f'RMS Energy: {rms:.4f} | Spectral Centroid: {centroid:.1f} Hz', fontsize=10)
        plt.tight_layout()
        plt.show()

interactive = widgets.interactive(update_plot, freq=freq_slider, noise=noise_slider)
display(widgets.VBox([freq_slider, noise_slider, output]))
update_plot(440.0, 0.05)